In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# models a continuous-time signal x(t) using a python function
class ContinuousSignal:
  def __init__(self, func):
    self.func = func

  def shift(self, shift):
    # returns x(t - shift)
    return ContinuousSignal(lambda t: self.func(t - shift))

  def add(self, other):
    # returns x(t) + y(t)
    return ContinuousSignal(lambda t: self.func(t) + other.func(t))

  def multiply(self, other):
    # returns x(t) * y(t)
    return ContinuousSignal(lambda t: self.func(t) * other.func(t))

  def multiply_const_factor(self, scaler):
    # returns a * x(t)
    return ContinuousSignal(lambda t: scaler * self.func(t))

  def plot(self, t_min=-3, t_max=3, num_points=1000, title=""):
    # np.linspace makes num_points evenly spaced values from t_min to t_max
    t = np.linspace(t_min, t_max, num_points)
    x = self.func(t)

    plt.plot(t, x)
    plt.title(title)
    plt.xlabel("t (Time)")
    plt.ylabel("x(t)")
    plt.grid(True)

In [ ]:
# time window is [-INF, INF]
INF = 3

# continuous-time LTI system with a given impulse response h(t)
class LTI_Continuous:
  def __init__(self, impulse_response):
    self.impulse_response = impulse_response

  def linear_combination_of_impulses(self, input_signal, delta):
    # rectangular impulse: height 1/delta for 0 <= t < delta, 0 otherwise
    # the tiny 1e-9 shrinks the right edge so floating point rounding of k*delta
    # can never make two neighbouring impulses overlap at a shared edge point
    def rect_impulse(t):
      # np.logical_and does elementwise "and" of two boolean arrays
      return np.logical_and(t >= 0, t < delta - 1e-9) / delta

    base = ContinuousSignal(rect_impulse)

    impulses = []
    coefficients = []

    # impulses are placed at t_k = k * delta covering [-INF, INF]
    # np.ceil rounds up to the nearest integer
    k_max = int(np.ceil(INF / delta))
    for k in range(-k_max, k_max + 1):
      t_k = k * delta
      # impulse_k(t) = delta_impulse(t - t_k)
      impulses.append(base.shift(t_k))
      # coefficient c_k = x(t_k) * delta
      coefficients.append(input_signal.func(t_k) * delta)

    return impulses, coefficients

  def output_approx(self, input_signal, delta):
    # not required in this practice
    raise NotImplementedError

In [ ]:
# builds x_hat(t) = sum of c_k * impulse_k(t)
def reconstruct(impulses, coefficients):
  # np.zeros_like gives an array of zeros with the same shape as t
  s = ContinuousSignal(lambda t: np.zeros_like(t))
  for imp, c in zip(impulses, coefficients):
    # component signal x_k(t) = c_k * impulse_k(t)
    s = s.add(imp.multiply_const_factor(c))
  return s

In [ ]:
def main():
  # folder where all plots are saved
  os.makedirs("continuous_practice", exist_ok=True)

  # ---------- step 1: create the system and the input signal ----------

  # input signal x(t) = e^(-t) * u(t)
  def x_func(t):
    # np.exp computes e^t elementwise
    return np.exp(-t) * (t >= 0)

  x = ContinuousSignal(x_func)

  # impulse response h(t) = u(t), not used for output in this practice
  h = ContinuousSignal(lambda t: (t >= 0) * 1.0)
  system = LTI_Continuous(h)

  t = np.linspace(-INF, INF, 1000)

  # ---------- figure 1: input signal ----------
  plt.figure()
  x.plot(-INF, INF, 1000, title=f"x(t), INF = {INF}")
  plt.savefig("continuous_practice/figure1_input_signal.png")
  plt.show()

  # ---------- step 2 + figure 2: decompose with delta = 0.5 ----------
  delta = 0.5
  impulses, coefficients = system.linear_combination_of_impulses(x, delta)
  x_hat = reconstruct(impulses, coefficients)

  k_max = len(impulses) // 2
  n_plots = len(impulses) + 1  # all components plus the reconstruction
  cols = 3
  rows = (n_plots + cols - 1) // cols

  fig, axes = plt.subplots(rows, cols, figsize=(12, 2.5 * rows), sharex=True, sharey=True)
  # flatten turns the 2d grid of axes into a 1d array
  axes = axes.flatten()

  for i, (imp, c) in enumerate(zip(impulses, coefficients)):
    k = i - k_max
    # component signal x_k(t) = c_k * impulse_k(t)
    component = imp.multiply_const_factor(c)
    axes[i].plot(t, component.func(t))
    axes[i].set_title(f"$\\delta(t - ({k}\\Delta)) x({k}\\Delta) \\Delta$")
    axes[i].grid(True)

  # last used subplot shows the reconstructed signal
  axes[len(impulses)].plot(t, x_hat.func(t))
  axes[len(impulses)].set_title("Reconstructed Signal")
  axes[len(impulses)].grid(True)

  # hide leftover empty subplots
  for j in range(n_plots, len(axes)):
    axes[j].axis("off")

  fig.suptitle("Impulses multiplied by coefficients")
  fig.tight_layout()
  fig.savefig("continuous_practice/figure2_impulses_and_reconstruction.png")
  plt.show()

  # ---------- figure 3: reconstruction with varying delta ----------
  deltas = [0.5, 0.1, 0.05, 0.01]
  fig, axes = plt.subplots(2, 2, figsize=(12, 8))
  axes = axes.flatten()

  for ax, d in zip(axes, deltas):
    impulses, coefficients = system.linear_combination_of_impulses(x, d)
    x_hat = reconstruct(impulses, coefficients)

    ax.plot(t, x_hat.func(t), label="Reconstructed")
    ax.plot(t, x.func(t), label="x(t)")
    ax.set_title(f"$\\Delta$ = {d}")
    ax.set_xlabel("t (Time)")
    ax.set_ylabel("x(t)")
    ax.legend()
    ax.grid(True)

  fig.tight_layout()
  fig.savefig("continuous_practice/figure3_varying_delta.png")
  plt.show()

In [ ]:
main()